# 08 · Evaluación temporal final · noviembre-diciembre de 2022

Este notebook realiza la **evaluación final confirmatoria** del modelo XGBoost congelado y de la regresión logística de referencia. Los dos modelos y sus preprocesadores se cargan directamente desde los artefactos creados por el notebook 07.

La evaluación se limita a las filas etiquetadas de noviembre y diciembre de 2022 con `dataset_split == 'test'`. Aquí no se reentrena, no se ajusta ningún transformador y no se modifica ningún hiperparámetro.

## Protocolo metodológico

- **Entrenamiento ya realizado:** 2019, 2021 y enero-septiembre de 2022.
- **Validación ya realizada:** octubre de 2022.
- **Test temporal final:** noviembre-diciembre de 2022.
- **Año 2020:** excluido del modelo principal.
- **Año 2023:** no se lee; queda reservado para análisis descriptivo de viajes.

XGBoost ya fue seleccionado antes de este test. Por eso las métricas de noviembre-diciembre no se utilizan para volver a elegir modelo, ajustar umbrales ni cambiar variables. La logística se conserva únicamente como línea base interpretable.

## Entorno, rutas y bloqueo contra repeticiones

La marca `_EVALUACION_FINAL_COMPLETADA.txt` se crea al terminar correctamente. Si se vuelve a ejecutar el notebook después de completar el test, el código se detendrá para evitar consultas repetidas accidentales.

In [ ]:
# %pip install pandas numpy scikit-learn xgboost matplotlib

from pathlib import Path
import gc
import json
import pickle
import platform
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
try:
    import xgboost
except Exception as error:
    raise RuntimeError(
        'XGBoost no puede cargarse en este kernel. Selecciona el mismo kernel '
        'con el que ejecutaste el notebook 07.'
    ) from error
from IPython.display import display
from sklearn.metrics import (
    ConfusionMatrixDisplay, accuracy_score, balanced_accuracy_score,
    classification_report, confusion_matrix, f1_score, log_loss,
)

# Localizamos el proyecto tanto si el notebook se abre desde la raíz como desde notebooks/.
CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == 'notebooks' else CURRENT_DIR
MODEL_DATA_DIR = PROJECT_ROOT / 'notebooks' / 'Datos modelado'
FEATURES_DIR = MODEL_DATA_DIR / 'estacion_hora_features'
ARTIFACTS_DIR = MODEL_DATA_DIR / 'validacion_externa'
OUTPUT_DIR = MODEL_DATA_DIR / 'test_final_2022'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Parámetros fijos del protocolo final.
TARGET = 'risk_class_1h'
TIME_COLUMN = 'fecha_hora_local'
TEST_PERIODS = ('202211', '202212')
CLASS_NAMES = {0: 'estable', 1: 'riesgo_vaciado', 2: 'riesgo_saturacion'}
CHUNK_SIZE = 100_000
PREDICTION_BATCH_SIZE = 100_000
ALLOW_RERUN = False
COMPLETION_SENTINEL = OUTPUT_DIR / '_EVALUACION_FINAL_COMPLETADA.txt'

# Este bloqueo solo se activa después de una ejecución completa, no después de un fallo parcial.
if COMPLETION_SENTINEL.exists() and not ALLOW_RERUN:
    raise RuntimeError(
        'La evaluación final ya figura como completada. No la repitas. '
        'Consulta los CSV guardados en test_final_2022.'
    )

print('Datos de modelado:', MODEL_DATA_DIR)
print('Resultados finales:', OUTPUT_DIR)
CURRENT_VERSIONS = {
    'python': platform.python_version(),
    'pandas': pd.__version__,
    'scikit-learn': sklearn.__version__,
    'xgboost': xgboost.__version__,
}
print('Versiones:', CURRENT_VERSIONS)

# Los objetos pickle deben abrirse con las mismas versiones usadas por el notebook 07.
EXPECTED_MODEL_VERSIONS = {'scikit-learn': '1.9.0', 'xgboost': '3.2.0'}
version_mismatches = {
    package: {'expected': expected, 'current': CURRENT_VERSIONS[package]}
    for package, expected in EXPECTED_MODEL_VERSIONS.items()
    if CURRENT_VERSIONS[package] != expected
}
if version_mismatches:
    raise RuntimeError(
        'Selecciona el mismo kernel que ejecutó el notebook 07 antes de abrir el test. '
        f'Versiones incompatibles: {version_mismatches}'
    )

## Barreras temporales y selección física de archivos

Solo se seleccionan las particiones `202211` y `202212`. Las aserciones detienen el proceso si falta un mes, aparece otro periodo o algún archivo de 2023 entra por error.

In [ ]:
def period_from_file(file_path: Path) -> str:
    # El periodo se encuentra al final del nombre: estacion_hora_features_YYYYMM.csv.
    return file_path.stem.rsplit('_', maxsplit=1)[-1]

# Se construyen explícitamente las dos rutas del test; no se cargan train ni validation.
test_files = [
    FEATURES_DIR / f'estacion_hora_features_{period}.csv'
    for period in TEST_PERIODS
]

if not all(path.exists() for path in test_files):
    missing = [str(path) for path in test_files if not path.exists()]
    raise FileNotFoundError(f'Faltan particiones del test final: {missing}')

assert tuple(period_from_file(path) for path in test_files) == TEST_PERIODS
assert all(period.startswith('2022') for period in TEST_PERIODS)
assert not any(period.startswith('2023') for period in TEST_PERIODS)

print('Únicos archivos que se leerán como test:')
for path in test_files:
    print(' -', path.name)

## Carga de modelos y preprocesadores congelados

Cada `.pkl` contiene el estimador, el preprocesador ya ajustado con train, la lista exacta de variables y la configuración congelada. En esta sección únicamente se cargan y auditan esos objetos.

In [ ]:
# Primero verificamos la auditoría que acompañaba a los modelos del notebook 07.
audit_07_path = ARTIFACTS_DIR / 'auditoria_validacion_externa.csv'
if not audit_07_path.exists():
    raise FileNotFoundError('Falta la auditoría del notebook 07.')
audit_07 = pd.read_csv(audit_07_path, encoding='utf-8-sig')
audit_07_values = dict(zip(audit_07['control'], audit_07['value'].astype(str)))
assert audit_07_values['ultimo_mes_train'] == '202209'
assert audit_07_values['mes_validacion_externa'] == '202210'
assert audit_07_values['validacion_usada_en_fit'].lower() == 'false'
assert audit_07_values['noviembre_diciembre_leidos'].lower() == 'false'
assert audit_07_values['test_2023_consultado'].lower() == 'false'
assert audit_07_values['modelo_seleccionado'] == 'xgboost_refined'

artifact_paths = {
    'xgboost_refined': ARTIFACTS_DIR / 'xgboost_refined_validacion_externa.pkl',
    'logistic_refined': ARTIFACTS_DIR / 'logistic_refined_validacion_externa.pkl',
}

# Cargamos los artefactos sin llamar a fit ni modificar sus parámetros.
artifacts = {}
for model_name, artifact_path in artifact_paths.items():
    if not artifact_path.exists():
        raise FileNotFoundError(f'No se encontró el artefacto congelado: {artifact_path}')
    with artifact_path.open('rb') as file:
        artifacts[model_name] = pickle.load(file)

required_keys = {'model', 'preprocessor', 'features', 'classes', 'frozen_configuration'}
for model_name, artifact in artifacts.items():
    assert required_keys.issubset(artifact), f'Artefacto incompleto: {model_name}'

# Los dos modelos deben usar exactamente las mismas variables y clases.
MODEL_FEATURES = list(artifacts['xgboost_refined']['features'])
assert MODEL_FEATURES == list(artifacts['logistic_refined']['features'])
assert artifacts['xgboost_refined']['classes'] == CLASS_NAMES
assert artifacts['logistic_refined']['classes'] == CLASS_NAMES

# Comprobamos también que el XGBoost cargado conserva los 381 árboles congelados.
xgb_configuration = artifacts['xgboost_refined']['frozen_configuration']
assert int(xgb_configuration['n_estimators']) == 381
assert int(artifacts['xgboost_refined']['model'].get_params()['n_estimators']) == 381

artifact_audit = pd.DataFrame([
    {
        'model': model_name,
        'estimator_class': type(artifact['model']).__name__,
        'preprocessor_class': type(artifact['preprocessor']).__name__,
        'n_features_raw': len(artifact['features']),
        'artifact_path': str(artifact_paths[model_name]),
    }
    for model_name, artifact in artifacts.items()
])
display(artifact_audit)
print('Configuración XGBoost congelada:')
print(json.dumps(xgb_configuration, indent=2, ensure_ascii=False))

## Carga exclusiva del test etiquetado

Se leen únicamente las columnas necesarias de noviembre-diciembre. Las filas sin etiqueta se excluyen porque no permiten evaluar, y todas las observaciones restantes deben estar marcadas como `test`.

In [ ]:
METADATA_COLUMNS = [TIME_COLUMN, 'station_id']
READ_COLUMNS = list(dict.fromkeys(
    MODEL_FEATURES + METADATA_COLUMNS + [TARGET, 'dataset_split']
))

# Guardamos contadores de auditoría para documentar qué se excluye.
raw_rows = 0
rows_wrong_split = 0
rows_without_target = 0
test_parts = []

for file_path in test_files:
    for chunk in pd.read_csv(
        file_path, usecols=READ_COLUMNS, chunksize=CHUNK_SIZE, low_memory=False,
    ):
        raw_rows += len(chunk)
        rows_wrong_split += int((~chunk['dataset_split'].eq('test')).sum())
        rows_without_target += int(chunk[TARGET].isna().sum())

        # Solo las filas de test con etiqueta real entran en las métricas.
        eligible = chunk.loc[
            chunk['dataset_split'].eq('test') & chunk[TARGET].notna()
        ].copy()
        if not eligible.empty:
            eligible[TARGET] = eligible[TARGET].astype('int8')
            test_parts.append(eligible)

if not test_parts:
    raise ValueError('No se encontraron filas etiquetadas en el test final.')

test = pd.concat(test_parts, ignore_index=True)
test[TIME_COLUMN] = pd.to_datetime(test[TIME_COLUMN], errors='raise')

# Controles de integridad temporal y de etiqueta.
assert rows_wrong_split == 0
assert test['dataset_split'].eq('test').all()
assert test[TARGET].notna().all()
assert set(test[TARGET].unique()).issubset(CLASS_NAMES)
assert test[TIME_COLUMN].dt.to_period('M').astype(str).isin(['2022-11', '2022-12']).all()
assert test[TIME_COLUMN].min() >= pd.Timestamp('2022-11-01 00:00:00')
assert test[TIME_COLUMN].max() < pd.Timestamp('2023-01-01 00:00:00')
assert not test.duplicated([TIME_COLUMN, 'station_id']).any()

class_distribution = (
    test[TARGET].value_counts().sort_index().rename('rows').to_frame()
)
class_distribution['class_name'] = class_distribution.index.map(CLASS_NAMES)
class_distribution['share'] = class_distribution['rows'] / len(test)

print(f'Filas físicas leídas: {raw_rows:,}')
print(f'Filas sin etiqueta excluidas: {rows_without_target:,}')
print(f'Filas evaluables: {len(test):,}')
print('Rango temporal:', test[TIME_COLUMN].min(), '→', test[TIME_COLUMN].max())
display(class_distribution)

del test_parts
gc.collect()

## Transformación y predicción sin reentrenamiento

Los preprocesadores se usan únicamente mediante `transform`, y los estimadores mediante `predict` y `predict_proba`. El procesamiento por lotes reduce el consumo de memoria sin alterar las predicciones.

In [ ]:
X_test_raw = test[MODEL_FEATURES]
y_test = test[TARGET].astype(int).to_numpy()

def predict_in_batches(artifact: dict, raw_features: pd.DataFrame, batch_size: int):
    # Esta función no ajusta nada: transforma y predice lotes consecutivos.
    prediction_batches = []
    probability_batches = []
    start_time = time.perf_counter()

    for start in range(0, len(raw_features), batch_size):
        stop = min(start + batch_size, len(raw_features))
        raw_batch = raw_features.iloc[start:stop]
        transformed_batch = artifact['preprocessor'].transform(raw_batch).astype(np.float32)
        prediction_batches.append(artifact['model'].predict(transformed_batch).astype(int))
        probability_batches.append(artifact['model'].predict_proba(transformed_batch))
        del raw_batch, transformed_batch
        gc.collect()

    elapsed_seconds = time.perf_counter() - start_time
    predictions = np.concatenate(prediction_batches)
    probabilities = np.vstack(probability_batches)
    return predictions, probabilities, elapsed_seconds

model_outputs = {}
for model_name in ('xgboost_refined', 'logistic_refined'):
    prediction, probabilities, elapsed_seconds = predict_in_batches(
        artifacts[model_name], X_test_raw, PREDICTION_BATCH_SIZE
    )
    assert len(prediction) == len(test)
    assert probabilities.shape == (len(test), 3)
    assert np.allclose(probabilities.sum(axis=1), 1.0, atol=1e-5)
    model_outputs[model_name] = {
        'prediction': prediction,
        'probabilities': probabilities,
        'prediction_seconds': elapsed_seconds,
    }
    print(f'{model_name}: {len(prediction):,} predicciones en {elapsed_seconds:.1f} s')

## Métricas finales y rendimiento por clase

F1 macro es la métrica principal y balanced accuracy la secundaria. También se incluyen métricas globales, Brier multiclase y calibración por confianza. La tabla no vuelve a seleccionar un ganador: XGBoost ya estaba fijado antes de abrir el test.

In [ ]:
def multiclass_brier_score(y_true: np.ndarray, probabilities: np.ndarray) -> float:
    # El Brier multiclase compara las probabilidades con la codificación one-hot real.
    one_hot = np.eye(3, dtype=float)[y_true]
    return float(np.mean(np.sum((probabilities - one_hot) ** 2, axis=1)))

def expected_calibration_error(
    y_true: np.ndarray, prediction: np.ndarray, probabilities: np.ndarray, n_bins: int = 15,
) -> float:
    # ECE resume la diferencia entre confianza media y frecuencia real de acierto.
    confidence = probabilities.max(axis=1)
    correct = (prediction == y_true).astype(float)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.minimum(np.digitize(confidence, edges[1:-1], right=True), n_bins - 1)
    ece = 0.0
    for bin_id in range(n_bins):
        mask = bin_ids == bin_id
        if mask.any():
            ece += mask.mean() * abs(correct[mask].mean() - confidence[mask].mean())
    return float(ece)

metric_rows = []
report_parts = []
confusion_parts = []
prediction_parts = []

for model_name, output in model_outputs.items():
    prediction = output['prediction']
    probabilities = output['probabilities']

    metric_rows.append({
        'model': model_name,
        'f1_macro': f1_score(y_test, prediction, average='macro'),
        'balanced_accuracy': balanced_accuracy_score(y_test, prediction),
        'f1_weighted': f1_score(y_test, prediction, average='weighted'),
        'accuracy': accuracy_score(y_test, prediction),
        'log_loss': log_loss(y_test, probabilities, labels=[0, 1, 2]),
        'brier_multiclass': multiclass_brier_score(y_test, probabilities),
        'ece_15_bins': expected_calibration_error(y_test, prediction, probabilities),
        'prediction_seconds': output['prediction_seconds'],
        'test_rows': len(test),
    })

    report = pd.DataFrame(classification_report(
        y_test, prediction, labels=[0, 1, 2],
        target_names=[CLASS_NAMES[index] for index in range(3)],
        output_dict=True, zero_division=0,
    )).T.reset_index(names='class')
    report.insert(0, 'model', model_name)
    report_parts.append(report)

    matrix = confusion_matrix(y_test, prediction, labels=[0, 1, 2])
    confusion_parts.append(pd.DataFrame([
        {
            'model': model_name, 'real': real, 'predicted': predicted,
            'real_name': CLASS_NAMES[real], 'predicted_name': CLASS_NAMES[predicted],
            'count': int(matrix[real, predicted]),
        }
        for real in range(3) for predicted in range(3)
    ]))

    model_predictions = test[METADATA_COLUMNS + [TARGET]].copy()
    model_predictions.insert(0, 'model', model_name)
    model_predictions['prediction'] = prediction
    model_predictions['is_error'] = prediction != y_test
    model_predictions['confidence'] = probabilities.max(axis=1)
    for class_value in range(3):
        model_predictions[f'probability_{class_value}'] = probabilities[:, class_value]
    prediction_parts.append(model_predictions)

metrics = pd.DataFrame(metric_rows)
reports = pd.concat(report_parts, ignore_index=True)
confusions = pd.concat(confusion_parts, ignore_index=True)
predictions = pd.concat(prediction_parts, ignore_index=True)

# Conservamos el orden metodológico: modelo final y después línea base.
metrics['model'] = pd.Categorical(
    metrics['model'], categories=['xgboost_refined', 'logistic_refined'], ordered=True
)
metrics = metrics.sort_values('model').reset_index(drop=True)
metrics['model'] = metrics['model'].astype(str)

display(metrics)
display(reports.loc[reports['class'].isin(CLASS_NAMES.values())])

## Estabilidad entre validación y test

La comparación con octubre permite medir deriva temporal. Las diferencias se describen, pero no se usan para modificar el modelo después de abrir noviembre-diciembre.

In [ ]:
validation_metrics_path = ARTIFACTS_DIR / 'comparacion_validacion_externa.csv'
if not validation_metrics_path.exists():
    raise FileNotFoundError('Faltan las métricas de octubre creadas por el notebook 07.')

# Unimos las métricas ya calculadas en octubre con las del test recién obtenido.
validation_metrics = pd.read_csv(validation_metrics_path, encoding='utf-8-sig')
comparison_columns = ['model', 'f1_macro', 'balanced_accuracy', 'f1_weighted', 'accuracy', 'log_loss']
validation_view = validation_metrics[comparison_columns].rename(columns={
    column: f'{column}_validation_october' for column in comparison_columns if column != 'model'
})
test_view = metrics[comparison_columns].rename(columns={
    column: f'{column}_test_november_december' for column in comparison_columns if column != 'model'
})
validation_test_comparison = validation_view.merge(test_view, on='model', validate='one_to_one')

# Una diferencia negativa en F1 o balanced accuracy indica caída respecto a octubre.
for metric_name in ('f1_macro', 'balanced_accuracy', 'f1_weighted', 'accuracy', 'log_loss'):
    validation_test_comparison[f'delta_test_minus_validation_{metric_name}'] = (
        validation_test_comparison[f'{metric_name}_test_november_december']
        - validation_test_comparison[f'{metric_name}_validation_october']
    )

display(validation_test_comparison)

In [ ]:
# Mostramos matrices normalizadas para comparar el recall de cada clase.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
for axis, model_name in zip(axes, ('xgboost_refined', 'logistic_refined')):
    ConfusionMatrixDisplay.from_predictions(
        y_test, model_outputs[model_name]['prediction'], labels=[0, 1, 2],
        display_labels=[CLASS_NAMES[index] for index in range(3)],
        normalize='true', values_format='.2f', cmap='Blues',
        ax=axis, colorbar=False,
    )
    axis.set_title(model_name.replace('_', ' ').title())
    axis.tick_params(axis='x', rotation=20)
fig.suptitle('Test temporal final · noviembre-diciembre de 2022')
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'matrices_confusion_test_final_2022.png', dpi=160, bbox_inches='tight')
plt.show()

# La figura principal destaca las métricas que guiaron el estudio.
metric_plot = metrics.set_index('model')[['f1_macro', 'balanced_accuracy']]
ax = metric_plot.plot(kind='bar', figsize=(8, 4.2), color=['#0B6E99', '#D97904'])
ax.set_ylim(0, 1)
ax.set_ylabel('Puntuación')
ax.set_title('Rendimiento en el test temporal final')
ax.tick_params(axis='x', rotation=0)
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
ax.figure.savefig(OUTPUT_DIR / 'comparacion_metricas_test_final_2022.png', dpi=160, bbox_inches='tight')
plt.show()

## Análisis descriptivo de errores por estación y hora

Estas tablas localizan los grupos difíciles sin cambiar el modelo. Para cada clase se registra soporte, recall, falsos negativos y falsos positivos; así se evita interpretar tasas calculadas sobre muy pocos casos.

In [ ]:
def grouped_error_table(model_predictions: pd.DataFrame, group_column: str) -> pd.DataFrame:
    # Calculamos métricas de error dentro de cada estación u hora.
    rows = []
    for group_value, group in model_predictions.groupby(group_column, dropna=False):
        actual = group[TARGET].to_numpy(dtype=int)
        predicted = group['prediction'].to_numpy(dtype=int)
        row = {
            group_column: group_value,
            'rows': len(group),
            'errors': int((actual != predicted).sum()),
            'error_rate': float((actual != predicted).mean()),
            'accuracy': float((actual == predicted).mean()),
        }
        for class_value in range(3):
            real_mask = actual == class_value
            predicted_mask = predicted == class_value
            support = int(real_mask.sum())
            true_positives = int((real_mask & predicted_mask).sum())
            row[f'support_class_{class_value}'] = support
            row[f'recall_class_{class_value}'] = (
                true_positives / support if support else np.nan
            )
            row[f'false_negatives_class_{class_value}'] = int((real_mask & ~predicted_mask).sum())
            row[f'false_positives_class_{class_value}'] = int((~real_mask & predicted_mask).sum())
        rows.append(row)
    return pd.DataFrame(rows)

error_by_station_parts = []
error_by_hour_parts = []

for model_name in ('xgboost_refined', 'logistic_refined'):
    model_predictions = predictions.loc[predictions['model'].eq(model_name)].copy()
    model_predictions['hour'] = pd.to_datetime(model_predictions[TIME_COLUMN]).dt.hour

    station_table = grouped_error_table(model_predictions, 'station_id')
    station_table.insert(0, 'model', model_name)
    error_by_station_parts.append(station_table)

    hour_table = grouped_error_table(model_predictions, 'hour')
    hour_table.insert(0, 'model', model_name)
    error_by_hour_parts.append(hour_table)

errors_by_station = pd.concat(error_by_station_parts, ignore_index=True)
errors_by_hour = pd.concat(error_by_hour_parts, ignore_index=True)

# Mostramos únicamente ejemplos con soporte suficiente para evitar rankings engañosos.
xgb_station_errors = errors_by_station.loc[
    errors_by_station['model'].eq('xgboost_refined') & (errors_by_station['rows'] >= 500)
].nlargest(10, 'errors')
xgb_hour_errors = errors_by_hour.loc[
    errors_by_hour['model'].eq('xgboost_refined')
].nlargest(10, 'error_rate')

print('Estaciones XGBoost con mayor número de errores, soporte ≥ 500:')
display(xgb_station_errors)
print('Horas XGBoost con mayor tasa de error:')
display(xgb_hour_errors)

## Calibración de probabilidades

Las curvas one-vs-rest comparan la probabilidad media asignada a cada riesgo con su frecuencia observada. Se incluyen como diagnóstico; no se calibran probabilidades ni se cambian umbrales usando el test.

In [ ]:
def calibration_table_for_model(
    model_name: str, y_true: np.ndarray, probabilities: np.ndarray, n_bins: int = 10,
) -> pd.DataFrame:
    # Usamos intervalos fijos para que las tablas de ambos modelos sean comparables.
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    rows = []
    for class_value in range(3):
        class_probability = probabilities[:, class_value]
        observed = (y_true == class_value).astype(float)
        bin_ids = np.minimum(
            np.digitize(class_probability, edges[1:-1], right=True), n_bins - 1
        )
        for bin_id in range(n_bins):
            mask = bin_ids == bin_id
            if not mask.any():
                continue
            mean_probability = float(class_probability[mask].mean())
            observed_frequency = float(observed[mask].mean())
            rows.append({
                'model': model_name,
                'class_value': class_value,
                'class_name': CLASS_NAMES[class_value],
                'bin': bin_id,
                'bin_lower': edges[bin_id],
                'bin_upper': edges[bin_id + 1],
                'rows': int(mask.sum()),
                'mean_probability': mean_probability,
                'observed_frequency': observed_frequency,
                'absolute_gap': abs(mean_probability - observed_frequency),
            })
    return pd.DataFrame(rows)

calibration = pd.concat([
    calibration_table_for_model(
        model_name, y_test, model_outputs[model_name]['probabilities']
    )
    for model_name in ('xgboost_refined', 'logistic_refined')
], ignore_index=True)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), sharex=True, sharey=True)
for class_value, axis in enumerate(axes):
    for model_name, color in [('xgboost_refined', '#0B6E99'), ('logistic_refined', '#D97904')]:
        subset = calibration.loc[
            calibration['model'].eq(model_name)
            & calibration['class_value'].eq(class_value)
        ]
        axis.plot(
            subset['mean_probability'], subset['observed_frequency'],
            marker='o', label=model_name, color=color,
        )
    axis.plot([0, 1], [0, 1], '--', color='gray', linewidth=1)
    axis.set_title(CLASS_NAMES[class_value])
    axis.set_xlabel('Probabilidad media predicha')
    axis.grid(alpha=0.2)
axes[0].set_ylabel('Frecuencia observada')
axes[-1].legend(loc='lower right')
fig.suptitle('Calibración descriptiva en el test temporal final')
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'calibracion_test_final_2022.png', dpi=160, bbox_inches='tight')
plt.show()

## Exportación reproducible y auditoría final

Se guardan resultados tabulares, predicciones y controles metodológicos. La marca de finalización se escribe en último lugar: si una celda anterior falla, la evaluación no se considera completada.

In [ ]:
# Exportamos tablas planas en UTF-8 con BOM para facilitar su apertura en Excel.
metrics.to_csv(
    OUTPUT_DIR / 'comparacion_test_final_2022.csv', index=False, encoding='utf-8-sig'
)
reports.to_csv(
    OUTPUT_DIR / 'informe_por_clase_test_final_2022.csv', index=False, encoding='utf-8-sig'
)
confusions.to_csv(
    OUTPUT_DIR / 'matrices_confusion_test_final_2022.csv', index=False, encoding='utf-8-sig'
)
predictions.to_csv(
    OUTPUT_DIR / 'predicciones_test_final_2022.csv', index=False, encoding='utf-8-sig'
)
errors_by_station.to_csv(
    OUTPUT_DIR / 'errores_por_estacion_test_final_2022.csv', index=False, encoding='utf-8-sig'
)
errors_by_hour.to_csv(
    OUTPUT_DIR / 'errores_por_hora_test_final_2022.csv', index=False, encoding='utf-8-sig'
)
calibration.to_csv(
    OUTPUT_DIR / 'calibracion_test_final_2022.csv', index=False, encoding='utf-8-sig'
)
class_distribution.reset_index(names='class_value').to_csv(
    OUTPUT_DIR / 'distribucion_clases_test_final_2022.csv', index=False, encoding='utf-8-sig'
)
artifact_audit.to_csv(
    OUTPUT_DIR / 'artefactos_congelados_test_final_2022.csv', index=False, encoding='utf-8-sig'
)
validation_test_comparison.to_csv(
    OUTPUT_DIR / 'comparacion_validacion_vs_test_2022.csv', index=False, encoding='utf-8-sig'
)

# Esta auditoría hace explícitas las barreras temporales y la ausencia de ajuste.
audit_table = pd.DataFrame([
    {'control': 'ultimo_mes_train_segun_auditoria_07', 'value': audit_07_values['ultimo_mes_train']},
    {'control': 'mes_validacion_segun_auditoria_07', 'value': audit_07_values['mes_validacion_externa']},
    {'control': 'test_leido_por_notebook_07', 'value': audit_07_values['noviembre_diciembre_leidos']},
    {'control': 'periodos_test_leidos', 'value': ','.join(TEST_PERIODS)},
    {'control': 'dataset_split_exigido', 'value': 'test'},
    {'control': 'primer_timestamp_test', 'value': str(test[TIME_COLUMN].min())},
    {'control': 'ultimo_timestamp_test', 'value': str(test[TIME_COLUMN].max())},
    {'control': 'filas_fisicas_leidas', 'value': raw_rows},
    {'control': 'filas_sin_etiqueta_excluidas', 'value': rows_without_target},
    {'control': 'filas_evaluadas', 'value': len(test)},
    {'control': 'filas_fuera_split_test', 'value': rows_wrong_split},
    {'control': 'llamadas_fit_en_notebook', 'value': 0},
    {'control': 'preprocesadores_reajustados', 'value': False},
    {'control': 'hiperparametros_modificados', 'value': False},
    {'control': 'n_estimators_xgboost', 'value': int(xgb_configuration['n_estimators'])},
    {'control': 'modelo_final_preestablecido', 'value': 'xgboost_refined'},
    {'control': 'linea_base_preestablecida', 'value': 'logistic_refined'},
    {'control': 'datos_2023_leidos', 'value': False},
])
audit_table.to_csv(
    OUTPUT_DIR / 'auditoria_test_final_2022.csv', index=False, encoding='utf-8-sig'
)

# Guardamos un manifiesto legible por máquinas con las decisiones del experimento.
manifest = {
    'evaluation_name': 'test_temporal_final_noviembre_diciembre_2022',
    'final_model': 'xgboost_refined',
    'baseline_model': 'logistic_refined',
    'primary_metric': 'f1_macro',
    'secondary_metric': 'balanced_accuracy',
    'test_periods': list(TEST_PERIODS),
    'test_rows': int(len(test)),
    'fit_calls': 0,
    'artifacts': {name: str(path) for name, path in artifact_paths.items()},
}
with (OUTPUT_DIR / 'manifiesto_test_final_2022.json').open('w', encoding='utf-8') as file:
    json.dump(manifest, file, indent=2, ensure_ascii=False)

# La marca se crea solo cuando todas las exportaciones anteriores han terminado.
COMPLETION_SENTINEL.write_text(
    'Evaluación final completada. No modificar el modelo usando estos resultados.\n',
    encoding='utf-8',
)

display(audit_table)
print('Evaluación final completada y protegida contra repeticiones.')
print('Resultados guardados en:', OUTPUT_DIR)

## Cómo interpretar y cerrar esta fase

1. Reporta F1 macro y balanced accuracy como métricas principales del modelo final.
2. Compara XGBoost con la logística solo como referencia; no vuelvas a seleccionar modelos.
3. Examina especialmente recall y F1 de vaciado y saturación, junto con sus falsos negativos.
4. Explica que noviembre-diciembre había sido inspeccionado en experimentos preliminares y presenta esta salida como evaluación temporal final confirmatoria.
5. No vuelvas a ejecutar este notebook ni ajustes el modelo tras ver el test.

El siguiente paso es interpretar el XGBoost congelado y estudiar sus errores por variable, estación, hora y clase. Enero-febrero de 2023 se utilizará únicamente en un análisis descriptivo de viajes, sin crear disponibilidad ni etiquetas de riesgo artificiales.